In [2]:
# ── Colab Setup (skip automatically if running locally) ──────────────────────
import sys, os

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    # transformers provides HuggingFace PatchTST (fully multivariate)
    subprocess.run(['pip', 'install', 'transformers', '-q'], check=True)

    REPO_URL = 'https://github.com/WoodyChang21/ECE1508_GenAI.git'
    if not os.path.exists('/content/ECE1508_GenAI'):
        subprocess.run(
            ['git', 'clone', '--branch', 'Model', REPO_URL, '/content/ECE1508_GenAI'],
            check=True,
        )
    os.chdir('/content/ECE1508_GenAI/notebooks')

    os.makedirs('/content/ECE1508_GenAI/data/splits', exist_ok=True)
    os.makedirs('/content/ECE1508_GenAI/data/predictions', exist_ok=True)

    splits_present = all(
        os.path.exists(f'/content/ECE1508_GenAI/data/splits/{f}')
        for f in ['train.parquet', 'val.parquet', 'test.parquet']
    )
    if not splits_present:
        from google.colab import files as colab_files
        print('Upload train.parquet, val.parquet, test.parquet from your local data/splits/ folder:')
        uploaded = colab_files.upload()
        for fname, data in uploaded.items():
            dest = f'/content/ECE1508_GenAI/data/splits/{fname}'
            with open(dest, 'wb') as f:
                f.write(data)
            print(f'  Saved → {dest}')
    else:
        print('Data splits already present — skipping upload.')

    import torch
    if torch.cuda.is_available():
        print(f'\nGPU: {torch.cuda.get_device_name(0)}')
    else:
        print('\nNo GPU detected. Go to Runtime → Change runtime type → T4 GPU.')

print('Setup complete.')

Data splits already present — skipping upload.

GPU: Tesla T4
Setup complete.


In [ ]:
import gc
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import PatchTSTConfig, PatchTSTForPrediction
import matplotlib.pyplot as plt

from scripts.models.data_loader import HIST_EXOG_COLS
from scripts.models.metrics import compute_all

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

TARGET_COL = 'return_1h'
ALL_COLS   = [TARGET_COL] + HIST_EXOG_COLS   # 21 columns; return_1h is channel 0
CANDIDATES = [24, 60, 120, 240]
plt.rcParams['figure.figsize'] = (14, 4)

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────

def patch_params(input_size: int):
    patch_len = max(4, min(16, input_size // 4))
    stride    = max(2, patch_len // 2)
    return patch_len, stride


def gpu_mem_report(clear: bool = False) -> None:
    """Print current GPU memory usage; optionally flush the CUDA cache."""
    if not torch.cuda.is_available():
        print('No GPU available.')
        return
    if clear:
        torch.cuda.empty_cache()
        gc.collect()
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free      = total - reserved
    print(f'GPU  allocated: {allocated:.2f} GiB | reserved: {reserved:.2f} GiB | '
          f'free: {free:.2f} GiB / {total:.2f} GiB'
          + ('  [cache cleared]' if clear else ''))


class ZScoreScaler:
    """Fit on training data, transform any split with the same statistics."""
    def fit(self, data: np.ndarray):
        self.mean_ = data.mean(axis=0)
        self.std_  = data.std(axis=0) + 1e-8
        return self

    def transform(self, data: np.ndarray) -> np.ndarray:
        return (data - self.mean_) / self.std_

    def inverse_col0(self, z: np.ndarray) -> np.ndarray:
        """De-normalise the target channel (index 0) back to original scale."""
        return z * self.std_[0] + self.mean_[0]


class WindowDataset(Dataset):
    """
    Produces (past_values, future_values) pairs for PatchTST training.
    past_values  : (context_length, 21)  — all channels, lookback window
    future_values: (1, 21)               — all channels at t+1 (used for MSE loss)
    """
    def __init__(self, data: np.ndarray, context_length: int):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.ctx  = context_length

    def __len__(self):
        return len(self.data) - self.ctx

    def __getitem__(self, i):
        return {
            'past_values':   self.data[i : i + self.ctx],
            'future_values': self.data[i + self.ctx : i + self.ctx + 1],
        }


def build_model(input_size: int) -> PatchTSTForPrediction:
    patch_len, stride = patch_params(input_size)
    config = PatchTSTConfig(
        num_input_channels=len(ALL_COLS),   # 21: return_1h + 20 features
        context_length=input_size,
        prediction_length=1,
        patch_length=patch_len,
        stride=stride,
        d_model=128,
        num_attention_heads=8,
        num_hidden_layers=3,
        dropout=0.2,
        head_dropout=0.0,
        channel_attention=False,   # channel-independent mode (original paper)
        loss='mse',
    )
    return PatchTSTForPrediction(config)


def train_model(
    model: PatchTSTForPrediction,
    data: np.ndarray,
    context_length: int,
    max_steps: int,
    batch_size: int = 64,
) -> PatchTSTForPrediction:
    dataset = WindowDataset(data, context_length)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
    opt     = torch.optim.Adam(model.parameters(), lr=1e-4)
    scaler  = torch.cuda.amp.GradScaler(enabled=DEVICE == 'cuda')
    model.to(DEVICE).train()
    step = 0
    while step < max_steps:
        for batch in loader:
            pv  = batch['past_values'].to(DEVICE)
            fv  = batch['future_values'].to(DEVICE)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=DEVICE == 'cuda'):
                loss = model(past_values=pv, future_values=fv).loss
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            step += 1
            if step >= max_steps:
                break
    return model


def predict_walkforward(
    model: PatchTSTForPrediction,
    full_data: np.ndarray,
    context_length: int,
    start_idx: int,
    batch_size: int = 512,
) -> np.ndarray:
    """
    Batch walk-forward inference from start_idx to end of full_data.
    Builds all context windows upfront (no retraining), then runs inference
    in batches. Returns predicted values for the target channel (index 0)
    in normalised space.
    """
    n = len(full_data) - start_idx
    windows = np.stack([
        full_data[i - context_length : i]
        for i in range(start_idx, len(full_data))
    ])  # (n, context_length, 21)

    model.eval()
    preds = []
    with torch.no_grad():
        for i in range(0, n, batch_size):
            x   = torch.tensor(windows[i : i + batch_size], dtype=torch.float32).to(DEVICE)
            with torch.cuda.amp.autocast(enabled=DEVICE == 'cuda'):
                out = model(past_values=x)
            # prediction_outputs: (batch, pred_len=1, num_channels=21)
            p   = out.prediction_outputs[:, 0, 0].float().cpu().numpy()   # target channel
            preds.append(p)
    return np.concatenate(preds)

In [5]:
# ── Load and normalise data ───────────────────────────────────────────────────
train_raw = pd.read_parquet('../data/splits/train.parquet')
val_raw   = pd.read_parquet('../data/splits/val.parquet')
test_raw  = pd.read_parquet('../data/splits/test.parquet')

for df in [train_raw, val_raw, test_raw]:
    df['is_first_bar'] = df['is_first_bar'].astype(float)

# Feature matrix: shape (n_rows, 21); return_1h is column 0 (the target channel)
train_mat = train_raw[ALL_COLS].values.astype(np.float32)
val_mat   = val_raw[ALL_COLS].values.astype(np.float32)
test_mat  = test_raw[ALL_COLS].values.astype(np.float32)

# Fit scaler on train only — no leakage
scaler     = ZScoreScaler().fit(train_mat)
train_norm = scaler.transform(train_mat)
val_norm   = scaler.transform(val_mat)
test_norm  = scaler.transform(test_mat)

trainval_norm = np.concatenate([train_norm, val_norm], axis=0)
full_norm     = np.concatenate([train_norm, val_norm, test_norm], axis=0)

n_train, n_val, n_test = len(train_norm), len(val_norm), len(test_norm)

print(f'Train: {n_train:,}  Val: {n_val:,}  Test: {n_test:,}')
print(f'Input channels ({len(ALL_COLS)}): {ALL_COLS}')

Train: 21,028  Val: 1,744  Test: 2,469
Input channels (21): ['return_1h', 'open', 'high', 'low', 'close', 'volume', 'return_4h', 'return_24h', 'is_first_bar', 'vol_24h', 'vol_60h', 'volume_ratio', 'rsi_14', 'macd', 'macd_signal', 'macd_diff', 'bb_upper', 'bb_lower', 'bb_width', 'vix_log', 'vix_change_1h']


In [ ]:
# ── Lookback tuning — val MAE per candidate ───────────────────────────────────
# Train on train, walk-forward predict val, pick best input_size.
# Val predictions from the best model are saved for empirical interval calibration.

val_maes        = {}
val_preds_cache = {}

for input_size in CANDIDATES:
    patch_len, stride = patch_params(input_size)
    model = build_model(input_size)
    model = train_model(model, train_norm, input_size, max_steps=500)

    z_preds = predict_walkforward(model, trainval_norm, input_size, start_idx=n_train)
    z_true  = val_norm[:, 0]
    val_mae = float(np.mean(np.abs(z_preds - z_true)))

    val_maes[input_size]        = val_mae
    val_preds_cache[input_size] = z_preds
    print(f'  input_size={input_size:3d}  patch_len={patch_len}  stride={stride}  val MAE={val_mae:.6f}')
    del model
    gc.collect()
    torch.cuda.empty_cache()

best_input_size = min(val_maes, key=val_maes.get)
print(f'\nBest input_size: {best_input_size}  (val MAE={val_maes[best_input_size]:.6f})')

# Val errors used later for empirical prediction intervals
z_val_true  = val_norm[:, 0]
z_val_preds = val_preds_cache[best_input_size]
val_errors  = z_val_true - z_val_preds   # error = true − pred

In [ ]:
# ── Final training on train + val ────────────────────────────────────────────
final_model = build_model(best_input_size)
final_model = train_model(final_model, trainval_norm, best_input_size, max_steps=1000)
print(f'Final model trained on {len(trainval_norm):,} rows (train + val), input_size={best_input_size}')

In [ ]:
# ── Walk-forward test predictions + metrics ───────────────────────────────────
z_test_preds = predict_walkforward(final_model, full_norm, best_input_size, start_idx=n_train + n_val)
z_test_true  = test_norm[:, 0]

# Empirical prediction intervals from val errors.
# val_error = true − pred, so:
#   lower bound = pred + q10(error)   (true is above this 90% of the time)
#   upper bound = pred + q90(error)   (true is below this 90% of the time)
q = {lvl: float(np.quantile(val_errors, lvl)) for lvl in [0.05, 0.10, 0.90, 0.95]}

z_lo_80 = z_test_preds + q[0.10]
z_hi_80 = z_test_preds + q[0.90]
z_lo_90 = z_test_preds + q[0.05]
z_hi_90 = z_test_preds + q[0.95]

# De-normalise back to original return_1h scale
y_true = scaler.inverse_col0(z_test_true)
y_pred = scaler.inverse_col0(z_test_preds)
lo_80  = scaler.inverse_col0(z_lo_80)
hi_80  = scaler.inverse_col0(z_hi_80)
lo_90  = scaler.inverse_col0(z_lo_90)
hi_90  = scaler.inverse_col0(z_hi_90)

results = compute_all(y_true, y_pred, lo_80, hi_80, lo_90, hi_90)

print('=== PatchTST Test Results (multivariate — 21 channels) ===')
print(f"  RMSE              : {results['rmse']:.6f}")
print(f"  MAE               : {results['mae']:.6f}")
print(f"  Directional Acc   : {results['dir_acc']:.4f}")
print(f"  Coverage 80%      : {results['coverage_80']:.4f}  (target: 0.80)")
print(f"  Coverage 90%      : {results['coverage_90']:.4f}  (target: 0.90)")
print(f"  Sharpe Ratio      : {results['sharpe']:.4f}")
print(f"  Max Drawdown      : {results['max_drawdown']:.6f}")

In [ ]:
# ── Plot: predicted vs actual, first 200 test bars ────────────────────────────
n = 200
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(range(n), y_true[:n], label='Actual return_1h',      alpha=0.8, linewidth=0.8, color='black')
ax.plot(range(n), y_pred[:n], label='PatchTST (predicted)',  alpha=0.8, linewidth=0.8, color='darkorange')
ax.fill_between(range(n), lo_90[:n], hi_90[:n], alpha=0.12, color='darkorange', label='90% interval')
ax.fill_between(range(n), lo_80[:n], hi_80[:n], alpha=0.22, color='darkorange', label='80% interval')
ax.axhline(0, color='gray', linewidth=0.5)
ax.legend(fontsize=9)
ax.set_title(f'PatchTST (HuggingFace, {len(ALL_COLS)} channels): predicted vs actual — first 200 test bars (2024)')
ax.set_xlabel('Test bar index')
ax.set_ylabel('return_1h')
plt.tight_layout()
plt.show()

In [ ]:
# ── Save predictions ──────────────────────────────────────────────────────────
preds_df = pd.DataFrame({
    'ds':       np.arange(n_test),
    'datetime': test_raw['datetime'].values[:n_test],
    'y':        y_true,
    'pred':     y_pred,
    'lo_80':    lo_80,
    'hi_80':    hi_80,
    'lo_90':    lo_90,
    'hi_90':    hi_90,
    'model':    'PatchTST',
})
preds_df.to_parquet('../data/predictions/patchtst_preds.parquet', index=False)
print(f'Saved {len(preds_df):,} rows → data/predictions/patchtst_preds.parquet')
print(preds_df.head(3))